# Hydrogeological and Geospatial Data for Surface and Subsurface Flow Modelling in Swedish Coastal Catchments Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and explore a dataset described with a Croissant schema using the `mlcroissant` library.

The workflow follows a step-by-step structure:
- load the Croissant metadata,
- inspect dataset-level information,
- discover record sets and fields using their `@id` values,
- extract records into pandas DataFrames,
- perform exploratory data analysis (EDA), and
- create a few general-purpose visualizations.

### Dataset Source
Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.8kx9-mx2j/fair2.json`

> Important: in this notebook, all dataset entities are referenced by their Croissant `@id` values whenever available.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
We begin by loading the Croissant dataset from its schema URL. We also inspect the dataset-level metadata in a safe way.

Note that `dataset.metadata` is handled as a single metadata object. We avoid treating it like a plain dictionary object directly.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint

sns.set_theme(style="whitegrid")

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.8kx9-mx2j/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Convert metadata to a JSON-like object only when needed for inspection
metadata_json = metadata.to_json()

print("Dataset loaded successfully.\n")
print("Name:", getattr(metadata, "name", "N/A"))
print("Identifier:", metadata_json.get("identifier", "N/A"))
print("Version:", metadata_json.get("version", "N/A"))
print("License:", metadata_json.get("license", "N/A"))
print("Spatial coverage:", metadata_json.get("spatialCoverage", "N/A"))
print("Date published:", metadata_json.get("datePublished", "N/A"))

description = metadata_json.get("description", "")
print("\nDescription preview:")
print(description[:1200] + ("..." if len(description) > 1200 else ""))

## 2. Dataset Overview
In this section we inspect the Croissant structure.

We will:
- list available top-level metadata keys,
- inspect record sets declared in the Croissant file,
- collect record set `@id`s dynamically, and
- inspect fields for each record set when present.

Because Croissant datasets can vary in structure, the notebook uses variables and helper functions to adapt dynamically.

In [ ]:
# Show the top-level keys available in the JSON serialization of the metadata
top_level_keys = sorted(metadata_json.keys())
print("Top-level metadata keys:")
print(top_level_keys)

# Extract record set metadata entries from the JSON representation
record_sets_meta = metadata_json.get("recordSet", []) or []
print(f"\nNumber of declared record sets in metadata_json['recordSet']: {len(record_sets_meta)}")

if len(record_sets_meta) == 0:
    print("No record sets were explicitly listed in the provided metadata snapshot.")
    print("We will still attempt discovery from the loaded Croissant dataset object where possible.")

In [ ]:
def summarize_record_set(rs):
    """Return a compact summary of a record set metadata object represented as a dict."""
    return {
        "@id": rs.get("@id", "N/A"),
        "name": rs.get("name", "N/A"),
        "description": (rs.get("description", "")[:120] + "...") if rs.get("description") else "",
        "field_count": len(rs.get("field", []) or []),
    }

if record_sets_meta:
    record_set_summary = pd.DataFrame([summarize_record_set(rs) for rs in record_sets_meta])
    display(record_set_summary)
else:
    record_set_summary = pd.DataFrame(columns=["@id", "name", "description", "field_count"])
    display(record_set_summary)

### 2.1 Record set and field IDs
The next cell extracts record set IDs and field IDs using their `@id` values.

If the metadata snapshot does not include explicit record sets, the code remains safe and continues gracefully.

In [ ]:
# Collect record set IDs dynamically from the metadata JSON
record_set_ids = [rs.get("@id") for rs in record_sets_meta if rs.get("@id")]
print("Record set @id values discovered:")
print(record_set_ids if record_set_ids else "[]")

# Build a field inventory keyed by record set @id
field_inventory = {}
for rs in record_sets_meta:
    rs_id = rs.get("@id")
    fields = rs.get("field", []) or []
    field_inventory[rs_id] = [
        {
            "field_@id": field.get("@id", "N/A"),
            "name": field.get("name", "N/A"),
            "description": field.get("description", ""),
            "dataType": field.get("dataType", "N/A"),
            "source": field.get("source", "N/A"),
        }
        for field in fields
    ]

if field_inventory:
    for rs_id, fields in field_inventory.items():
        print(f"\nRecord set: {rs_id}")
        if fields:
            display(pd.DataFrame(fields))
        else:
            print("No explicit fields listed for this record set.")
else:
    print("No field inventory available from the metadata snapshot.")

## 3. Data Extraction
We now attempt to read records from each record set into pandas DataFrames.

This section is written defensively because some Croissant datasets expose metadata cleanly but may not expose records in all runtime environments. Any loading error is caught and reported without stopping the notebook.

In [ ]:
dataframes = {}
record_load_report = []

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        record_load_report.append({
            "record_set_@id": record_set_id,
            "loaded": True,
            "rows": len(df),
            "columns": len(df.columns),
            "error": ""
        })
    except Exception as e:
        record_load_report.append({
            "record_set_@id": record_set_id,
            "loaded": False,
            "rows": 0,
            "columns": 0,
            "error": str(e)
        })

load_report_df = pd.DataFrame(record_load_report)
display(load_report_df)

In [ ]:
# Preview loaded DataFrames
if dataframes:
    for record_set_id, df in dataframes.items():
        print(f"\nRecord set @id: {record_set_id}")
        print(f"Shape: {df.shape}")
        print("Columns:")
        print(df.columns.tolist())
        display(df.head())
else:
    print("No record sets were successfully loaded into DataFrames.")
    print("This can happen if the dataset metadata is available but the record content is not directly exposed through the current endpoint or runtime.")

### 3.1 Example: inspect a single record set by `@id`
The next cell shows a reusable pattern for selecting one record set dynamically and printing a few raw records.

This follows the template style while keeping the notebook robust.

In [ ]:
example_record_set_id = record_set_ids[0] if record_set_ids else None

if example_record_set_id is not None:
    print(f"Example record set @id: {example_record_set_id}")
    try:
        for i, x in enumerate(dataset.records(record_set=example_record_set_id)):
            print(x)
            if i >= 2:
                break
    except Exception as e:
        print("Could not iterate over records for the example record set.")
        print("Reason:", e)
else:
    print("No record set @id is available to demonstrate record iteration.")

## 4. Exploratory Data Analysis (EDA)
We now perform generic EDA on whichever record sets were loaded successfully.

Because field layouts may differ across Croissant datasets, we:
- detect numeric columns automatically,
- select one loaded record set dynamically,
- filter records using a quantile-based threshold,
- normalize a numeric field, and
- optionally group by a categorical field if one exists.

All selections are based on actual loaded data rather than hard-coded column names.

In [ ]:
selected_record_set_id = None
selected_df = None

for rs_id, df in dataframes.items():
    if not df.empty:
        selected_record_set_id = rs_id
        selected_df = df.copy()
        break

if selected_df is None:
    print("No non-empty DataFrame is available for EDA.")
else:
    print("Selected record set for EDA:", selected_record_set_id)
    print("Shape:", selected_df.shape)
    display(selected_df.head())

In [ ]:
if selected_df is not None:
    # Try coercing object columns to numeric where possible to improve downstream analysis
    working_df = selected_df.copy()
    for col in working_df.columns:
        if working_df[col].dtype == "object":
            converted = pd.to_numeric(working_df[col], errors="ignore")
            working_df[col] = converted

    numeric_columns = working_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_columns = working_df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    print("Numeric columns:", numeric_columns)
    print("Categorical-like columns:", categorical_columns)

    if numeric_columns:
        display(working_df[numeric_columns].describe().T)
    else:
        print("No numeric columns detected in the selected record set.")

In [ ]:
filtered_df = None
normalized_field_name = None
numeric_field_id = None
group_field_id = None

if selected_df is not None:
    working_df = selected_df.copy()
    for col in working_df.columns:
        if working_df[col].dtype == "object":
            working_df[col] = pd.to_numeric(working_df[col], errors="ignore")

    numeric_columns = working_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_columns = working_df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        series = working_df[numeric_field_id].dropna()

        if not series.empty:
            threshold = series.quantile(0.50)
            filtered_df = working_df[working_df[numeric_field_id] > threshold].copy()
            print(f"Using numeric field: {numeric_field_id}")
            print(f"Threshold (median): {threshold}")
            print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")
            display(filtered_df.head())

            std = filtered_df[numeric_field_id].std()
            if pd.notna(std) and std != 0:
                normalized_field_name = f"{numeric_field_id}_normalized"
                filtered_df[normalized_field_name] = (
                    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
                    filtered_df[numeric_field_id].std()
                )
                print(f"Normalized field created: {normalized_field_name}")
                display(filtered_df[[numeric_field_id, normalized_field_name]].head())
            else:
                print("Standard deviation is zero or undefined; normalization was skipped.")

            if categorical_columns:
                group_field_id = categorical_columns[0]
                print(f"Attempting grouped summary by: {group_field_id}")
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean(numeric_only=False).reset_index()
                display(grouped_df.head(10))
            else:
                print("No categorical field available for grouping.")
        else:
            print("Selected numeric field has no non-null values.")
    else:
        print("No numeric field available for filtering and normalization.")

### 4.1 Missing values and duplicate checks
These checks are often useful during dataset preparation.

In [ ]:
if selected_df is not None:
    missing_summary = selected_df.isna().sum().sort_values(ascending=False).to_frame(name="missing_values")
    missing_summary["missing_fraction"] = missing_summary["missing_values"] / len(selected_df)
    print("Missing value summary:")
    display(missing_summary.head(20))

    duplicate_count = selected_df.duplicated().sum()
    print(f"Duplicate rows in selected record set ({selected_record_set_id}): {duplicate_count}")
else:
    print("No DataFrame available for missing value or duplicate checks.")

## 5. Visualization
We create a few flexible visualizations for the selected record set.

The code is adaptive:
- if at least one numeric field exists, we plot a histogram,
- if at least two numeric fields exist, we plot a scatter plot,
- if a grouping field was identified, we show a bar chart of grouped means.

In [ ]:
if selected_df is not None:
    viz_df = selected_df.copy()
    for col in viz_df.columns:
        if viz_df[col].dtype == "object":
            viz_df[col] = pd.to_numeric(viz_df[col], errors="ignore")

    numeric_columns = viz_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_columns = viz_df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    if len(numeric_columns) >= 1:
        plt.figure(figsize=(8, 4))
        sns.histplot(viz_df[numeric_columns[0]].dropna(), kde=True, bins=30)
        plt.title(f"Distribution of {numeric_columns[0]}\nRecord set: {selected_record_set_id}")
        plt.xlabel(numeric_columns[0])
        plt.ylabel("Count")
        plt.tight_layout()
        plt.show()
    else:
        print("No numeric columns available for histogram visualization.")

    if len(numeric_columns) >= 2:
        plt.figure(figsize=(6, 5))
        sns.scatterplot(data=viz_df, x=numeric_columns[0], y=numeric_columns[1], alpha=0.7)
        plt.title(f"{numeric_columns[0]} vs {numeric_columns[1]}\nRecord set: {selected_record_set_id}")
        plt.tight_layout()
        plt.show()
    else:
        print("At least two numeric columns are needed for a scatter plot.")

    if filtered_df is not None and group_field_id is not None and numeric_field_id is not None:
        grouped_plot_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id]
            .mean(numeric_only=False)
            .sort_values(ascending=False)
            .head(10)
            .reset_index()
        )
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_plot_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Top grouped mean values of {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
else:
    print("No DataFrame available for visualization.")

## 6. Dataset-Level Notes
Even when record-level data are not fully accessible, Croissant metadata remains valuable for understanding the dataset.

Below we summarize a few important dataset-level characteristics directly from metadata.

In [ ]:
dataset_level_summary = {
    "dataset_@id": metadata_json.get("@id", "N/A"),
    "name": metadata_json.get("name", "N/A"),
    "identifier": metadata_json.get("identifier", "N/A"),
    "version": metadata_json.get("version", "N/A"),
    "license": metadata_json.get("license", "N/A"),
    "citeAs": metadata_json.get("citeAs", "N/A"),
    "spatialCoverage": metadata_json.get("spatialCoverage", "N/A"),
    "keywords": metadata_json.get("keywords", []),
    "distribution_count": len(metadata_json.get("distribution", []) or []),
    "author_count": len(metadata_json.get("author", []) or []),
    "citation_count": len(metadata_json.get("citation", []) or []),
}

pprint(dataset_level_summary)

## 7. Conclusion
In this notebook, we:
- loaded a dataset from its Croissant schema URL,
- inspected dataset metadata safely through the `mlcroissant` metadata object,
- discovered record sets and fields by their `@id` values when available,
- attempted to load records into pandas DataFrames,
- ran reusable EDA steps on the first non-empty loaded record set, and
- created adaptive visualizations based on the available columns.

This pattern is useful for working with Croissant-described datasets whose structure may vary across repositories or endpoints.

For this dataset specifically, the metadata indicates a hydrogeological and geospatial collection focused on Swedish coastal catchments, with emphasis on reproducible modelling inputs, open formats, and FAIR²-oriented documentation.

### Next steps
- Inspect any successfully loaded record sets more deeply.
- Map field `@id` values to domain-specific variable names.
- Add geospatial plotting if coordinate columns are exposed in loaded records.
- Use the distribution metadata to locate underlying files for advanced analysis.